<a href="https://colab.research.google.com/github/neet813/SmartOps-Task-Platform/blob/main/SmartOps_Task__pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the libraries we need
# Run this first every time you open Colab

!pip install gspread google-auth pandas --quiet
print("✅ Libraries installed")

In [ ]:

# All imports in one place so nothing is missing later

import gspread
from google.oauth2.service_account import Credentials
from datetime import datetime, timedelta
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import uuid
import time
import threading

print("✅ All libraries imported successfully")

In [ ]:

# Upload your service account JSON file to Colab before running
# Then replace the filename below with your actual filename

scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    "your-service-account-file.json",  # ← replace with your filename
    scopes=scope
)

client = gspread.authorize(creds)
sheet  = client.open("SmartOps Task Platform")

print("✅ Connected to Google Sheet!")
print()
print("Tabs found:")
for ws in sheet.worksheets():
    print(f"   - {ws.title}")

In [ ]:

# Fill in your own details before running
# Never share this file publicly with real credentials

EMAIL_ADDRESS          = "your_gmail@gmail.com"   # your Gmail address
EMAIL_PASSWORD         = "your_app_password"       # your Gmail app password
MANAGER_EMAIL          = "manager@gmail.com"       # who gets escalation alerts

ESCALATION_HOURS       = 24   # hours before a task is considered overdue
CHECK_INTERVAL_MINUTES = 30   # how often the system checks automatically

print("✅ Settings loaded")
print(f"   Sending emails from  : {EMAIL_ADDRESS}")
print(f"   Manager alerts go to : {MANAGER_EMAIL}")
print(f"   Tasks escalate after : {ESCALATION_HOURS} hours")
print(f"   Auto-check every     : {CHECK_INTERVAL_MINUTES} minutes")

In [ ]:

# Run this once only when setting up for the first time
# WARNING: This clears all existing data in all tabs
# Do NOT run again after your system has real data

# Form Submissions tab — where new tasks arrive from the web form
form_tab = sheet.worksheet("Form Submissions")
form_tab.clear()
form_tab.append_row([
    "Timestamp", "Submitter Name", "Submitter Email",
    "Site", "Shift Manager", "Problem Description",
    "Action Needed", "Priority Score", "Status"
])

# Active Tasks tab — where all live tasks are tracked
active_tab = sheet.worksheet("Active Tasks")
active_tab.clear()
active_tab.append_row([
    "Task ID", "Timestamp", "Submitter Name", "Submitter Email",
    "Site", "Shift Manager", "Problem Description",
    "Action Needed", "Priority Score", "Priority Label",
    "Status", "Deadline", "Escalated", "Original Task ID"
])

# Archive tab — where completed tasks are stored
archive_tab = sheet.worksheet("Archive")
archive_tab.clear()
archive_tab.append_row([
    "Task ID", "Timestamp", "Submitter Name", "Submitter Email",
    "Site", "Shift Manager", "Problem Description",
    "Action Needed", "Priority Score", "Priority Label",
    "Resolved At", "Resolution Time (Hours)", "Closed By"
])

# Escalation Log tab — every escalation recorded here
escalation_tab = sheet.worksheet("Escalation Log")
escalation_tab.clear()
escalation_tab.append_row([
    "Escalation ID", "Original Task ID", "Task Description",
    "Site", "Submitter Email", "Manager Email",
    "Hours Overdue", "Escalated At", "New Task ID"
])

print("✅ All 4 tabs set up with headers!")
print("Go check your Google Sheet — each tab should have one header row only")

In [ ]:

# The brain of the system
# Reads a task description and gives it a score from 1 to 100
# Higher score = more urgent = prioritised higher in the dashboard

PRIORITY_KEYWORDS = {
    # CRITICAL (90-100) — safety emergencies, needs immediate action
    "fire": 100, "emergency": 100, "flood": 100,
    "gas leak": 100, "injury": 100, "accident": 95,
    "electric": 95, "power": 95, "unsafe": 90, "danger": 90,

    # HIGH (70-89) — urgent but not life threatening
    "broken toilet": 85, "no water": 85, "heating": 80,
    "security": 80, "broken door": 75, "lock": 75,
    "leak": 75, "blocked": 70, "lift": 70, "elevator": 70,

    # MEDIUM (40-69) — affects work but can wait a few hours
    "pest": 65, "rat": 70, "broken light": 60,
    "lighting": 55, "wifi": 50, "internet": 50,
    "cleaning": 50, "smell": 50, "computer": 45,
    "noise": 45, "printer": 40,

    # LOW (1-39) — minor issues, no urgency
    "window": 30, "kitchen": 25, "parking": 20,
    "bin": 20, "blind": 20, "chair": 15, "desk": 15,
    "coffee": 10,
}

def get_priority_score(description):
    # Check description against every keyword
    # Return the highest matching score found
    description_lower = description.lower()
    highest_score = 10  # minimum score if no keywords match

    for keyword, score in PRIORITY_KEYWORDS.items():
        if keyword in description_lower:
            if score > highest_score:
                highest_score = score

    return highest_score

def get_priority_label(score):
    # Convert number score into a readable label
    if score >= 90:
        return "🔴 CRITICAL"
    elif score >= 70:
        return "🟠 HIGH"
    elif score >= 40:
        return "🟡 MEDIUM"
    else:
        return "🟢 LOW"

def generate_task_id():
    # Creates a unique ID for each task e.g. TASK-B17CDE6F
    return "TASK-" + str(uuid.uuid4())[:8].upper()

def generate_escalation_id():
    # Creates a unique ID for each escalation e.g. ESC-A1B2C3D4
    return "ESC-" + str(uuid.uuid4())[:8].upper()

print("✅ Priority scoring engine ready")

In [ ]:

# Three email functions:
# 1. send_email             — base function, sends any email
# 2. send_completion_email  — sent to submitter when task is resolved
# 3. send_escalation_email  — sent to manager when task is overdue

def send_email(to_email, subject, body):
    # Base function — all emails go through here
    try:
        msg = MIMEMultipart()
        msg['From']    = EMAIL_ADDRESS
        msg['To']      = to_email
        msg['Subject'] = subject
        msg.attach(MIMEText(body, 'html'))

        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.send_message(msg)
        server.quit()
        print(f"  📧 Email sent to {to_email}")
        return True
    except Exception as e:
        print(f"  ❌ Email failed: {e}")
        return False

def send_completion_email(task):
    # Sent to the person who submitted the task when it is marked DONE
    subject = f"✅ Your task has been resolved — {task['Task ID']}"
    body = f"""
    <html><body>
    <h2 style="color:#2e7d32;">✅ Task Resolved</h2>
    <p>Hi {task['Submitter Name']},</p>
    <p>Your task has been successfully resolved.</p>
    <table border="1" cellpadding="8" style="border-collapse:collapse;">
        <tr><td><b>Task ID</b></td><td>{task['Task ID']}</td></tr>
        <tr><td><b>Problem</b></td><td>{task['Problem Description']}</td></tr>
        <tr><td><b>Site</b></td><td>{task['Site']}</td></tr>
        <tr><td><b>Priority</b></td><td>{task['Priority Label']}</td></tr>
        <tr><td><b>Resolved At</b></td><td>{datetime.now().strftime('%Y-%m-%d %H:%M')}</td></tr>
    </table>
    <p>Thank you for reporting this issue.</p>
    <p style="color:#888;">SmartOps Task Intelligence Platform</p>
    </body></html>
    """
    send_email(task['Submitter Email'], subject, body)

def send_escalation_email(task, hours_overdue, new_task_id):
    # Sent to the manager when a task has not been resolved within 24 hours
    subject = f"🚨 ESCALATION ALERT — Task overdue {hours_overdue}h — {task['Task ID']}"
    body = f"""
    <html><body>
    <h2 style="color:#c62828;">🚨 Escalation Alert</h2>
    <p>Hi Manager,</p>
    <p>The following task has been unresolved for <b>{hours_overdue} hours</b>
    and has been automatically escalated.</p>
    <table border="1" cellpadding="8" style="border-collapse:collapse;">
        <tr><td><b>Original Task ID</b></td><td>{task['Task ID']}</td></tr>
        <tr><td><b>New Escalation ID</b></td><td>{new_task_id}</td></tr>
        <tr><td><b>Problem</b></td><td>{task['Problem Description']}</td></tr>
        <tr><td><b>Site</b></td><td>{task['Site']}</td></tr>
        <tr><td><b>Priority</b></td><td>{task['Priority Label']}</td></tr>
        <tr><td><b>Submitted At</b></td><td>{task['Timestamp']}</td></tr>
        <tr><td><b>Hours Overdue</b></td><td style="color:red;"><b>{hours_overdue} hours</b></td></tr>
        <tr><td><b>Submitted By</b></td><td>{task['Submitter Name']}</td></tr>
    </table>
    <p>A new escalated task has been automatically created in the system.</p>
    <p style="color:#888;">SmartOps Task Intelligence Platform</p>
    </body></html>
    """
    send_email(MANAGER_EMAIL, subject, body)

print("✅ Email system ready")

In [ ]:
# Four functions that power the entire system
# These are called automatically by the scheduler every 30 minutes

def process_new_submissions():
    # Reads Form Submissions tab
    # Scores each new task, creates it in Active Tasks, marks it processed
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Checking for new submissions...")

    form_tab   = sheet.worksheet("Form Submissions")
    active_tab = sheet.worksheet("Active Tasks")
    all_rows   = form_tab.get_all_records()

    new_count = 0
    for row in all_rows:
        if row.get("Status") == "Processed":  continue
        if not row.get("Problem Description"): continue

        score    = get_priority_score(row["Problem Description"])
        label    = get_priority_label(score)
        task_id  = generate_task_id()
        deadline = (datetime.now() + timedelta(hours=ESCALATION_HOURS)).strftime('%Y-%m-%d %H:%M')

        active_tab.append_row([
            task_id,
            row.get("Timestamp", datetime.now().strftime('%Y-%m-%d %H:%M')),
            row.get("Submitter Name", ""),
            row.get("Submitter Email", ""),
            row.get("Site", ""),
            row.get("Shift Manager", ""),
            row.get("Problem Description", ""),
            row.get("Action Needed", ""),
            score, label, "OPEN", deadline, "NO", ""
        ])

        row_index = all_rows.index(row) + 2
        form_tab.update_cell(row_index, 9, "Processed")
        time.sleep(1)

        new_count += 1
        print(f"  ✅ {task_id} | {label} | {row['Problem Description'][:45]}")

    print(f"  📋 {new_count} new task(s) added." if new_count else "  ℹ️ No new submissions found.")

def check_overdue_tasks():
    # Reads Active Tasks tab
    # Any OPEN task past its deadline is escalated automatically
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Checking for overdue tasks...")

    active_tab     = sheet.worksheet("Active Tasks")
    escalation_tab = sheet.worksheet("Escalation Log")
    all_rows       = active_tab.get_all_records()

    escalation_count = 0
    for i, row in enumerate(all_rows):
        if row.get("Status") != "OPEN":    continue
        if row.get("Escalated") == "YES":  continue
        if not row.get("Deadline"):        continue

        try:
            deadline = datetime.strptime(row["Deadline"], '%Y-%m-%d %H:%M')
        except:
            continue

        now = datetime.now()
        if now > deadline:
            hours_overdue = round((now - deadline).total_seconds() / 3600, 1)
            new_task_id   = generate_task_id()
            esc_id        = generate_escalation_id()
            new_deadline  = (now + timedelta(hours=ESCALATION_HOURS)).strftime('%Y-%m-%d %H:%M')

            # Create new escalated task
            active_tab.append_row([
                new_task_id,
                now.strftime('%Y-%m-%d %H:%M'),
                row.get("Submitter Name", ""),
                row.get("Submitter Email", ""),
                row.get("Site", ""),
                row.get("Shift Manager", ""),
                f"🚨 ESCALATED: {row.get('Problem Description', '')}",
                row.get("Action Needed", ""),
                min(row.get("Priority Score", 50) + 20, 100),
                "🔴 CRITICAL", "OPEN", new_deadline, "YES",
                row.get("Task ID", "")
            ])

            # Log in Escalation Log
            escalation_tab.append_row([
                esc_id, row.get("Task ID", ""),
                row.get("Problem Description", ""),
                row.get("Site", ""),
                row.get("Submitter Email", ""),
                MANAGER_EMAIL, hours_overdue,
                now.strftime('%Y-%m-%d %H:%M'), new_task_id
            ])

            # Mark original task so it never escalates again
            row_index = i + 2
            active_tab.update_cell(row_index, 13, "YES")
            active_tab.update_cell(row_index, 11, "ESCALATED")
            time.sleep(1)

            send_escalation_email(row, hours_overdue, new_task_id)
            escalation_count += 1
            print(f"  🚨 {row['Task ID']} | Overdue {hours_overdue}h | New: {new_task_id}")

    print(f"  🚨 {escalation_count} task(s) escalated." if escalation_count else "  ✅ No overdue tasks found.")

def check_completed_tasks():
    # Reads Active Tasks tab
    # Any task marked DONE is moved to Archive and submitter gets an email
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Checking for completed tasks...")

    active_tab  = sheet.worksheet("Active Tasks")
    archive_tab = sheet.worksheet("Archive")
    all_rows    = active_tab.get_all_records()

    completed_count = 0
    rows_to_delete  = []

    for i, row in enumerate(all_rows):
        if row.get("Status") != "DONE": continue

        try:
            submitted        = datetime.strptime(row["Timestamp"], '%Y-%m-%d %H:%M')
            resolved_at      = datetime.now()
            resolution_hours = round((resolved_at - submitted).total_seconds() / 3600, 1)
        except:
            resolved_at      = datetime.now()
            resolution_hours = "N/A"

        archive_tab.append_row([
            row.get("Task ID", ""),
            row.get("Timestamp", ""),
            row.get("Submitter Name", ""),
            row.get("Submitter Email", ""),
            row.get("Site", ""),
            row.get("Shift Manager", ""),
            row.get("Problem Description", ""),
            row.get("Action Needed", ""),
            row.get("Priority Score", ""),
            row.get("Priority Label", ""),
            resolved_at.strftime('%Y-%m-%d %H:%M'),
            resolution_hours,
            row.get("Shift Manager", "")
        ])

        if row.get("Submitter Email"):
            send_completion_email(row)

        rows_to_delete.append(i + 2)
        completed_count += 1
        print(f"  ✅ {row['Task ID']} archived | Resolution time: {resolution_hours}h")

    # Delete in reverse order so row numbers stay correct
    for row_index in sorted(rows_to_delete, reverse=True):
        active_tab.delete_rows(row_index)
        time.sleep(0.5)

    print(f"  ✅ {completed_count} task(s) archived." if completed_count else "  ℹ️ No completed tasks found.")

def generate_recurring_tasks():
    # Auto-creates tasks on a schedule without anyone submitting them
    # Every Monday    — weekly safety check
    # Every 1st of month — monthly fire equipment inspection
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Checking recurring tasks...")

    active_tab = sheet.worksheet("Active Tasks")
    today      = datetime.now()
    recurring  = []

    if today.weekday() == 0:  # Monday
        recurring.append({
            "description": "🔄 Weekly Safety Check — All Areas",
            "action":      "Complete full site safety inspection checklist",
            "site":        "All Sites",
            "priority":    70
        })

    if today.day == 1:  # 1st of the month
        recurring.append({
            "description": "🔄 Monthly Fire Equipment Inspection",
            "action":      "Check all fire extinguishers, alarms, and exit signs",
            "site":        "All Sites",
            "priority":    85
        })

    for task in recurring:
        task_id  = generate_task_id()
        deadline = (today + timedelta(hours=ESCALATION_HOURS)).strftime('%Y-%m-%d %H:%M')
        score    = task["priority"]
        label    = get_priority_label(score)

        active_tab.append_row([
            task_id, today.strftime('%Y-%m-%d %H:%M'),
            "System", EMAIL_ADDRESS,
            task["site"], "Facilities Manager",
            task["description"], task["action"],
            score, label, "OPEN", deadline, "NO", "RECURRING"
        ])
        print(f"  🔄 {task_id} | {task['description'][:45]}")

    if not recurring:
        print("  ℹ️ No recurring tasks due today.")

print("✅ All four task processor functions ready")
print()
print("Functions loaded:")
print("  - process_new_submissions()")
print("  - check_overdue_tasks()")
print("  - check_completed_tasks()")
print("  - generate_recurring_tasks()")

In [ ]:
# Run this last — the system will run every 30 minutes automatically
# It will pick up new submissions, escalate overdue tasks,
# archive completed tasks, and generate recurring tasks
#
# IMPORTANT: Keep this Colab tab open
# If you close Colab the scheduler stops

def run_smartops():
    print()
    print("=" * 55)
    print(f"🚀 SmartOps AUTO RUN — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 55)

    try:
        process_new_submissions()
        time.sleep(10)
        check_overdue_tasks()
        time.sleep(10)
        check_completed_tasks()
        time.sleep(10)
        generate_recurring_tasks()
    except Exception as e:
        print(f"❌ System error: {e}")
        send_email(
            EMAIL_ADDRESS,
            "⚠️ SmartOps System Error",
            f"<html><body><p>Error detected: {str(e)}</p></body></html>"
        )

    print(f"⏰ Next run in {CHECK_INTERVAL_MINUTES} minutes...")
    print("=" * 55)

def scheduler_loop():
    while True:
        run_smartops()
        time.sleep(CHECK_INTERVAL_MINUTES * 60)

print("=" * 55)
print("🚀 Starting SmartOps Auto Scheduler...")
print("⏰ Runs every 30 minutes automatically")
print("📧 Emails fire automatically")
print("🚨 Escalations trigger automatically")
print("=" * 55)
print()

# Run once immediately then start background loop
run_smartops()

scheduler_thread = threading.Thread(target=scheduler_loop, daemon=True)
scheduler_thread.start()

print()
print("=" * 55)
print("✅ SmartOps is now fully live!")
print()
print("Keep this Colab tab open to keep it running.")
print("=" * 55)